In [1]:
import pandas as pd

In [5]:
pip install --upgrade transformers==4.40.2

     ---------------------------------------- 0.0/138.0 kB ? eta -:--:--
     -- ------------------------------------- 10.2/138.0 kB ? eta -:--:--
     -------- ---------------------------- 30.7/138.0 kB 435.7 kB/s eta 0:00:01
     --------------------- --------------- 81.9/138.0 kB 762.6 kB/s eta 0:00:01
     -------------------------------------- 138.0/138.0 kB 1.0 MB/s eta 0:00:00
   ---------------------------------------- 0.0/9.0 MB ? eta -:--:--
   - -------------------------------------- 0.4/9.0 MB 11.2 MB/s eta 0:00:01
   --- ------------------------------------ 0.7/9.0 MB 9.1 MB/s eta 0:00:01
   ---- ----------------------------------- 1.1/9.0 MB 8.5 MB/s eta 0:00:01
   ------ --------------------------------- 1.4/9.0 MB 8.3 MB/s eta 0:00:01
   -------- ------------------------------- 1.8/9.0 MB 8.3 MB/s eta 0:00:01
   --------- ------------------------------ 2.2/9.0 MB 8.2 MB/s eta 0:00:01
   ----------- ---------------------------- 2.5/9.0 MB 8.1 MB/s eta 0:00:01
   --------

  You can safely remove it manually.


In [5]:
pip install transformers[torch]


   ---------------------------------------- 0.0/354.7 kB ? eta -:--:--
   - -------------------------------------- 10.2/354.7 kB ? eta -:--:--
   ------------ --------------------------- 112.6/354.7 kB 2.2 MB/s eta 0:00:01
   ---------------------------------------  348.2/354.7 kB 4.3 MB/s eta 0:00:01
   ---------------------------------------- 354.7/354.7 kB 3.2 MB/s eta 0:00:00


In [4]:
pip install transformers datasets pandas scikit-learn torch

   ---------------------------------------- 0.0/10.4 MB ? eta -:--:--
   - -------------------------------------- 0.3/10.4 MB 8.6 MB/s eta 0:00:02
   -- ------------------------------------- 0.6/10.4 MB 8.0 MB/s eta 0:00:02
   --- ------------------------------------ 1.0/10.4 MB 7.9 MB/s eta 0:00:02
   ----- ---------------------------------- 1.4/10.4 MB 8.0 MB/s eta 0:00:02
   ------ --------------------------------- 1.8/10.4 MB 7.6 MB/s eta 0:00:02
   -------- ------------------------------- 2.2/10.4 MB 7.6 MB/s eta 0:00:02
   --------- ------------------------------ 2.5/10.4 MB 7.7 MB/s eta 0:00:02
   ----------- ---------------------------- 2.9/10.4 MB 7.8 MB/s eta 0:00:01
   ------------ --------------------------- 3.3/10.4 MB 7.8 MB/s eta 0:00:01
   ------------- -------------------------- 3.5/10.4 MB 7.7 MB/s eta 0:00:01
   -------------- ------------------------- 3.7/10.4 MB 7.2 MB/s eta 0:00:01
   --------------- ------------------------ 4.1/10.4 MB 7.2 MB/s eta 0:00:01
   ---

  You can safely remove it manually.


In [1]:
pip list | findstr transformers

transformers                      4.51.3
Note: you may need to restart the kernel to use updated packages.


In [1]:
from transformers import AutoTokenizer

## 데이터 로드

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split

# 수동 라벨 데이터 불러오기
df = pd.read_csv("데이터셋/수동라벨링.CSV", encoding='cp949')
df = df.drop_duplicates(subset='emotion_text').dropna(subset=['emotion_text', 'label'])

# 라벨 인코딩
label_map = {'negative': 0, 'neutral': 1, 'positive': 2}
df['label'] = df['label'].map(label_map)

# 학습/검증 데이터 분할
train_texts, val_texts, train_labels, val_labels = train_test_split(
    df['emotion_text'].tolist(),
    df['label'].tolist(),
    test_size=0.2,
    stratify=df['label'],
    random_state=42
)

print(f"Train size: {len(train_texts)}, Validation size: {len(val_texts)}")


Train size: 537, Validation size: 135


## 토크나이저 로딩 및 토큰화

In [2]:
from transformers import AutoTokenizer

# KcELECTRA 토크나이저 로드
tokenizer = AutoTokenizer.from_pretrained("beomi/KcELECTRA-base-v2022")

# 토큰화 함수 정의
def tokenize_function(texts, labels):
    tokenized = tokenizer(texts, padding=True, truncation=True, max_length=128)
    tokenized["label"] = labels
    return tokenized

# 학습/검증 데이터셋 토큰화
train_encodings = tokenize_function(train_texts, train_labels)
val_encodings = tokenize_function(val_texts, val_labels)

c:\Users\Admin\anaconda3\Lib\site-packages\huggingface_hub\file_download.py:943: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


## Dataset 변환 및 모델 로딩

In [3]:
import torch
from datasets import Dataset
from transformers import AutoModelForSequenceClassification

# Dataset 변환
train_dataset = Dataset.from_dict(train_encodings)
val_dataset = Dataset.from_dict(val_encodings)

# 모델 불러오기 (num_labels = 3 for 3-class classification)
model = AutoModelForSequenceClassification.from_pretrained(
    "beomi/KcELECTRA-base-v2022",
    num_labels=3
)

Some weights of ElectraForSequenceClassification were not initialized from the model checkpoint at beomi/KcELECTRA-base-v2022 and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


## Trainer 설정 및 학습

In [4]:
from transformers import Trainer, TrainingArguments

# 학습 설정
training_args = TrainingArguments(
    output_dir="./results",
    evaluation_strategy="epoch",
    save_strategy="epoch",
    num_train_epochs=5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    learning_rate=2e-5,
    weight_decay=0.01,
    load_best_model_at_end=True,
    logging_dir="./logs",
    logging_steps=50,
)

# Trainer 정의
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset
)

# 학습 시작!
trainer.train()

  0%|          | 0/170 [00:00<?, ?it/s]

c:\Users\Admin\anaconda3\Lib\site-packages\torch\utils\data\dataloader.py:665: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


  0%|          | 0/9 [00:00<?, ?it/s]

{'eval_loss': 0.7416277527809143, 'eval_runtime': 24.6827, 'eval_samples_per_second': 5.469, 'eval_steps_per_second': 0.365, 'epoch': 1.0}


c:\Users\Admin\anaconda3\Lib\site-packages\torch\utils\data\dataloader.py:665: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


...

c:\Users\Admin\anaconda3\Lib\site-packages\torch\utils\data\dataloader.py:665: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


{'loss': 0.3524, 'grad_norm': 1.6128450632095337, 'learning_rate': 2.3529411764705885e-06, 'epoch': 4.41}


  0%|          | 0/9 [00:00<?, ?it/s]

{'eval_loss': 0.6494259834289551, 'eval_runtime': 17.5114, 'eval_samples_per_second': 7.709, 'eval_steps_per_second': 0.514, 'epoch': 5.0}
{'train_runtime': 2344.3914, 'train_samples_per_second': 1.145, 'train_steps_per_second': 0.073, 'train_loss': 0.5337702695061178, 'epoch': 5.0}


TrainOutput(global_step=170, training_loss=0.5337702695061178, metrics={'train_runtime': 2344.3914, 'train_samples_per_second': 1.145, 'train_steps_per_second': 0.073, 'total_flos': 176614881649920.0, 'train_loss': 0.5337702695061178, 'epoch': 5.0})

In [5]:
# 세이브포인트 찍기
model.save_pretrained("./kcbert_model")
tokenizer.save_pretrained("./kcbert_model")

('./kcbert_model\\tokenizer_config.json',
 './kcbert_model\\special_tokens_map.json',
 './kcbert_model\\vocab.txt',
 './kcbert_model\\added_tokens.json',
 './kcbert_model\\tokenizer.json')

In [8]:
model.save_pretrained("./kcbert_emotion_model")
tokenizer.save_pretrained("./kcbert_emotion_model")

('./kcbert_emotion_model\\tokenizer_config.json',
 './kcbert_emotion_model\\special_tokens_map.json',
 './kcbert_emotion_model\\vocab.txt',
 './kcbert_emotion_model\\added_tokens.json',
 './kcbert_emotion_model\\tokenizer.json')

In [6]:
original_df = pd.read_csv("데이터셋/emotion_df.csv")
original_texts = original_df["emotion_text"].tolist()

In [9]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer

model = AutoModelForSequenceClassification.from_pretrained("./kcbert_emotion_model")
tokenizer = AutoTokenizer.from_pretrained("./kcbert_emotion_model")
model.eval()

ElectraForSequenceClassification(
  (electra): ElectraModel(
    (embeddings): ElectraEmbeddings(
      (word_embeddings): Embedding(54343, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): ElectraEncoder(
      (layer): ModuleList(
        (0-11): 12 x ElectraLayer(
          (attention): ElectraAttention(
            (self): ElectraSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): ElectraSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): L

In [10]:
import pandas as pd

df_full = pd.read_csv("데이터셋/emotion_df.csv")
texts_to_predict = df_full["emotion_text"].tolist()

## 감정 예측 함수 정의 & 실행


In [11]:
import torch

# label 인덱스를 감정 문자열로 변환하는 매핑
id2label = {0: 'negative', 1: 'neutral', 2: 'positive'}

# 예측 함수 정의
def predict_emotions(texts, batch_size=32):
    predicted_labels = []

    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        inputs = tokenizer(batch, padding=True, truncation=True, return_tensors="pt", max_length=128)
        with torch.no_grad():
            outputs = model(**inputs)
            logits = outputs.logits
            preds = torch.argmax(logits, dim=1).tolist()
            predicted_labels.extend(preds)

    return [id2label[label] for label in predicted_labels]

# 예측 실행
predicted_emotions = predict_emotions(texts_to_predict)

# 결과 붙이기
df_full["predicted_emotion"] = predicted_emotions

# 일부 결과 보기
df_full[["emotion_text", "predicted_emotion"]].head()

,emotion_text,predicted_emotion
0,시즌 끝났네… 이걸 또 하라고….?ㅠ 졸유함? 졸유? 졸유는 했지 이미 없었지만 이...,negative
1,면준 다들 면접준비 얼마나 하시나요 프리스타일 저 중요한곳 할 때는 주 넘게 햇어요...,negative
2,최종합격 후기 찾아보면 다들 너무 반짝 반짝 빛난다 저렇게 사니깐 합격하는구나 나같...,negative
3,인적성 인적성 단계에서 % 인적성 점수만 가지고 컷하나요..? 아님 서류도 다시 볼...,negative
4,선배님들~! 설문조사 참여 부탁드립니다! 안녕하세요! 중앙대학교 경영경제대학 학술제...,neutral


In [12]:
df_full.to_csv("감정분석_결과.csv", index=False, encoding="utf-8-sig")